## Johdanto 

Tässä oppitunnissa käydään läpi: 
- Mitä funktiokutsu on ja mihin sitä käytetään 
- Kuinka luoda funktiokutsu Azure OpenAI:n avulla 
- Kuinka integroida funktiokutsu sovellukseen 

## Oppimistavoitteet 

Oppitunnin suorittamisen jälkeen osaat ja ymmärrät: 

- Funktiokutsun käytön tarkoituksen 
- Funktiokutsun määrittämisen Azure Open AI -palvelussa 
- Kuinka suunnitella tehokkaita funktiokutsuja sovelluksesi käyttötarkoitukseen 


## Ymmärtäminen funktiokutsuista 

Tässä oppitunnissa haluamme rakentaa ominaisuuden koulutusstartupillemme, joka antaa käyttäjien käyttää chatbotia teknisten kurssien löytämiseen. Suosittelemme käyttäjän taitotason, nykyisen roolin ja kiinnostuksen kohteena olevan teknologian perusteella sopivia kursseja.

Tämän toteuttamiseksi käytämme yhdistelmää:
 - `Azure Open AI` luomaan käyttäjälle chat-kokemuksen
 - `Microsoft Learn Catalog API` auttamaan käyttäjiä löytämään kursseja käyttäjän pyynnön perusteella
 - `Function Calling` ottamaan käyttäjän kyselyn ja lähettämään sen funktiolle API-pyynnön tekemiseksi.

Aloittaaksemme katsotaan, miksi haluaisimme käyttää funktiokutsuja ylipäätään:


### Miksi funktion kutsuminen  

Jos olet suorittanut minkä tahansa muun tämän kurssin oppitunnin, ymmärrät todennäköisesti suurten kielimallien (LLM) käytön voiman. Toivottavasti näet myös joitakin niiden rajoituksia.  

Funktion kutsuminen on Azure Open AI -palvelun ominaisuus, joka auttaa voittamaan seuraavat rajoitukset:  
1) Johdonmukainen vastausmuoto  
2) Mahdollisuus käyttää sovelluksen muiden lähteiden dataa chat-kontekstissa  

Ennen funktion kutsumista LLM:n vastaukset olivat jäsentelemättömiä ja epäjohdonmukaisia. Kehittäjien piti kirjoittaa monimutkaista validointikoodia varmistaakseen, että he pystyivät käsittelemään kaikki vastausvaihtoehdot.  

Käyttäjät eivät voineet saada vastauksia kuten "Mikä on tämänhetkinen sää Tukholmassa?". Tämä johtui siitä, että mallien tiedot olivat rajoittuneet koulutusaikaan.  

Katsotaan alla olevaa esimerkkiä, joka havainnollistaa tätä ongelmaa:  

Sanotaan, että haluamme luoda opiskelijatietokannan, jotta voimme ehdottaa heille oikeaa kurssia. Alla on kaksi kuvausta opiskelijoista, jotka ovat hyvin samanlaisia tiedoiltaan.  


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Haluamme lähettää tämän LLM:lle, jotta se jäsentäisi tiedot. Tätä voidaan myöhemmin käyttää sovelluksessamme lähettämään se API:lle tai tallentamaan tietokantaan.

Luodaan kaksi identtistä kehotetta, joilla ohjaamme LLM:ää siitä, mistä tiedoista olemme kiinnostuneita:


Haluamme lähettää tämän LLM:lle jäsentämään ne osat, jotka ovat tärkeitä tuotteellemme. Joten voimme luoda kaksi identtistä kehotetta ohjaamaan LLM:ää: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Näiden kahden kehotteen luomisen jälkeen lähetämme ne LLM:lle käyttämällä `client.responses.create`. Tallennamme kehotteen `input`-muuttujaan ja annamme roolin `user`. Tämä jäljittelee käyttäjän kirjoittamaa viestiä chatbotille.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Nyt voimme lähettää molemmat pyynnöt LLM:lle ja tutkia saamaamme vastausta. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Vaikka kehotteet ovat samat ja kuvaukset samankaltaisia, voimme saada eri muotoja `Grades`-ominaisuudelle.

Jos ajat yllä olevan solun useita kertoja, muoto voi olla `3.7` tai `3.7 GPA`.

Tämä johtuu siitä, että LLM käsittelee jäsentämätöntä tietoa kirjoitetun kehotteen muodossa ja palauttaa myös jäsentämätöntä tietoa. Tarvitsemme kuitenkin jäsennellyn muodon, jotta tiedämme mitä odottaa tämän datan tallennuksessa tai käytössä.

Käyttämällä funktiokutsua voimme varmistaa, että saamme takaisin jäsenneltyä dataa. Käytettäessä funktiokutsua LLM ei itse asiassa suorita tai kutsu mitään funktioita. Sen sijaan luomme rakenteen, jota LLM noudattaa vastauksissaan. Käytämme sitten näitä jäsenneltyjä vastauksia tietääksemme, mitä funktiota sovelluksissamme ajetaan.
 


![Funktiokutsun virtauskaavio](../../../../translated_images/fi/Function-Flow.083875364af4f4bb.webp)


Voimme sitten ottaa funktion palauttaman arvon ja lähettää sen takaisin LLM:lle. LLM vastaa luonnollisella kielellä käyttäjän kyselyyn. 


### Käyttötapaukset funktiokutsujen käytössä

**Ulkoisten työkalujen kutsuminen**
Chatbotit ovat erinomaisia antamaan vastauksia käyttäjien kysymyksiin. Funktiokutsujen avulla chatbotit voivat käyttää käyttäjien viestejä tiettyjen tehtävien suorittamiseen. Esimerkiksi opiskelija voi pyytää bottia "Lähetä sähköposti ohjaajalleni, että tarvitsen enemmän apua tässä aiheessa". Tämä voi kutsua funktion `send_email(to: string, body: string)`


**API- tai tietokantakyselyiden luominen**
Käyttäjät voivat etsiä tietoa luonnollisella kielellä, joka muunnetaan muotoilluksi kyselyksi tai API-pyynnöksi. Esimerkkinä voi olla opettaja, joka tiedustelee "Ketkä opiskelijat ovat suorittaneet viimeisen tehtävän", mikä voisi kutsua funktion nimeltä `get_completed(student_name: string, assignment: int, current_status: string)`


**Rakenteellisen datan luominen**
Käyttäjät voivat hyödyntää tekstilohkoa tai CSV-tiedostoa ja käyttää LLM:ää tärkeän tiedon poimimiseen siitä. Esimerkiksi opiskelija voi muuntaa Wikipedia-artikkelin rauhansopimuksista AI-muistikorteiksi. Tämä voidaan tehdä käyttämällä funktiota nimeltä `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Ensimmäisen funktiokutsun luominen 

Funktiokutsun luomisprosessi sisältää 3 päävaihetta: 
1. Chat Completions -rajapinnan kutsuminen listalla omia funktioitasi ja käyttäjän viesti 
2. Mallin vastauksen lukeminen toiminnon suorittamiseksi, eli funktion tai API-kutsun suorittaminen 
3. Toisen kutsun tekeminen Chat Completions -rajapintaan funktiosi vastauksella käyttäjälle vastauksen luomiseksi. 


![Funktion kutsun kulku](../../../../translated_images/fi/LLM-Flow.3285ed8caf4796d7.webp)


### Funktion kutsun osat

#### Käyttäjän syöte

Ensimmäinen vaihe on luoda käyttäjän viesti. Tämä voidaan määritellä dynaamisesti ottamalla arvo tekstisyötteestä tai voit määrittää arvon tässä. Jos työskentelet Chat Completions API:n kanssa ensimmäistä kertaa, meidän on määriteltävä viestin `role` ja `content`.

`role` voi olla joko `system` (sääntöjen luominen), `assistant` (malli) tai `user` (loppukäyttäjä). Funktiokutsua varten asetamme tämän `user`:ksi ja esimerkkikysymyksen.


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Funktioiden luominen. 

Seuraavaksi määrittelemme funktion ja sen parametrit. Käytämme tässä vain yhtä funktiota nimeltä `search_courses`, mutta voit luoda useita funktioita.

**Tärkeää** : Funktiot sisältyvät järjestelmäviestiin LLM:lle, ja ne kuuluvat käytettävissä olevien tokenien määrään.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Määritelmät** 

`name` - Kutsuttavan funktion nimi. 

`description` - Kuvaus siitä, miten funktio toimii. Tässä on tärkeää olla tarkka ja selkeä. 

`parameters` - Lista arvoista ja muodosta, jonka haluat mallin tuottavan vastauksessaan. 


`type` - Ominaisuuksien tallennuksessa käytettävä tietotyyppi. 

`properties` - Lista erityisistä arvoista, joita malli käyttää vastauksessaan. 


`name` - Ominaisuuden nimi, jota malli käyttää muotoillussa vastauksessaan. 

`type` - Tämän ominaisuuden tietotyyppi. 

`description` - Kuvaus kyseisestä ominaisuudesta. 


**Valinnainen**

`required` - Pakollinen ominaisuus, jotta funktiokutsu voidaan suorittaa. 


### Funktion kutsun tekeminen 
Funktion määrittelyn jälkeen meidän täytyy nyt sisällyttää se kutsuun Chat Completion -sovellusliittymään. Teemme tämän lisäämällä `functions` pyyntöön. Tässä tapauksessa `functions=functions`. 

Vaihtoehtona on myös asettaa `function_call` arvoon `auto`. Tämä tarkoittaa, että annamme LLM:n päättää, mikä funktio pitäisi kutsua käyttäjän viestin perusteella sen sijaan, että määräisimme sen itse.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Tarkastellaanpa vastausta ja katsotaan, miten se on muotoiltu: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Voit nähdä, että funktion nimi on kutsuttu ja käyttäjän viestin perusteella LLM pystyi löytämään tiedot, jotka sopivat funktion argumentteihin. 


## 3. Funktion kutsujen integrointi sovellukseen. 


Kun olemme testanneet LLM:n muotoillun vastauksen, voimme nyt integroida sen sovellukseen. 

### Vuon hallinta 

Integroimista varten sovellukseemme teemme seuraavat vaiheet: 

Ensiksi teemme kutsun Open AI -palveluihin ja tallennamme viestin muuttujaan nimeltä `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Määrittelemme nyt funktion, joka kutsuu Microsoft Learn -rajapintaa saadakseen luettelon kursseista: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Parhaana käytäntönä tarkistamme sitten, haluaako malli kutsua funktiota. Tämän jälkeen luomme yhden käytettävissä olevista funktioista ja vastaamme sen siihen funktioon, jota kutsutaan. 
Otamme sen jälkeen funktion argumentit ja yhdistämme ne LLM:n argumentteihin.

Lopuksi lisäämme funktion kutsuviestin ja arvot, jotka `search_courses`-viesti palautti. Tämä antaa LLM:lle kaiken tarvittavan tiedon
vastatakseen käyttäjälle luonnollisella kielellä. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Nyt lähetämme päivitetyn viestin LLM:lle, jotta voimme saada luonnollisen kielen vastauksen API:n JSON-muotoisen vastauksen sijaan. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Koodhaaste 

Hienoa työtä! Jatkaaksesi Azure Open AI Function Callingin oppimista voit rakentaa: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Lisäparametreja funktiolle, jotka voivat auttaa oppijoita löytämään lisää kursseja. Löydät käytettävissä olevat API-parametrit täältä: 
 - Luo toinen funktiokutsu, joka ottaa oppijalta enemmän tietoa, kuten heidän äidinkielensä 
 - Luo virheenkäsittely, kun funktiokutsu ja/tai API-kutsu ei palauta sopivia kursseja 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
